# Lab 11

STAT 109: Simulating a population, drawing a sample, and asking what the
sample could learn

# Lab 11: Virtual worlds in R

> **Open in Google Colab (R)**
>
> [Open in
> Colab](https://colab.research.google.com/github/roverhol/stat109-public/blob/main/labs/lab-11.ipynb)
> — after it opens, use Runtime -\> Change runtime type and select R if
> needed.

## Introduction: why simulate data from a pretend population?

In this lab you will build a small virtual world (sometimes called a
digital twin of a study): you write down what is true in the entire
population, for example a mean $\mu$, a standard deviation $\sigma$, or
success probabilities $p$, $p_1$, and $p_2$—and then you use to draw a
random sample from that world. After that, you pretend you are a
resident of the world who only sees the sample. You run summaries and
tests (`mean()`, `t.test()`, `prop.test()`, …) exactly as you would on
real data.

The point of knowing the “ground truth” population is that you get **two
checks** you almost never get with real data:

1.  **Statistical method.** Did you pick a tool that matches the
    **variable types** and the **question** (for example a two-sample
    *t*-test for independent numerical groups, or `prop.test` for two
    proportions)? When you **already know** whether $H_0$ should hold
    (e.g. you built the world with $\mu_1 = \mu_2$ or with $p_1 = p_2$),
    you can see whether the conclusion from the sample is **usually**
    consistent with that truth. Sometimes a small *p*-value appears
    anyway: remember that Type I error? If we use a significance level
    of 0.05, then 5% of our samples will lead us to incorrectly reject
    the null hypothesis.

2.  **Correctness of code.** Bugs, wrong column names, reversed groups,
    or a mis-specified `prop.test` call can all produce output that
    looks fine but is actually incorrect . In a virtual world you can
    compare what the code inferred from the sample to what you coded as
    true for the population. If they disagree in a systematic way, the
    problem is likely your pipeline (or code suggested by a chat tool).

So the virtual world is a controlled place to validate both your choice
of method and your implementation in code.

Throughout this lab:

- **Population** quantities use Greek letters or symbols as in lecture
  (e.g. $\mu$, $\sigma$, $p$, $p_1$, $p_2$).
- **Sample** quantities are what you compute from the simulated rows
  (e.g. $\hat{p}=x/n$ as the proportion of successes in n trials in the
  dataset, $\bar{x}$ as the sample mean of a numerical variable from $n$
  items or people).

In [ ]:
library(tidyverse)

------------------------------------------------------------------------

## 1. Binary outcomes: `rbinom` and population proportion $p$

For each person $i$, let $Y_i = 1$ if a condition is present (“success”)
and $Y_i = 0$ otherwise. In the population, $p = P(Y_i = 1)$ is the
**proportion** (same as the **mean** of the 0/1 outcomes).

We simulate $n$ independent people with
**`rbinom(n, size = 1, prob = p)`**: each draw is one Bernoulli trial
with success probability **`p`** (the argument `prob` in R is that
population probability).

In [ ]:
set.seed(1101)

# Population (universe) you choose:
p <- 0.22 # P(Y = 1) for a randomly chosen person
n <- 400

y <- rbinom(n, size = 1, prob = p)

# Sample proportion (same as mean of 0/1):
p_hat <- mean(y)

tibble(
  population_parameter = p,
  sample_size = n,
  sample_proportion_p_hat = p_hat
)

**Read the output like a resident of the universe.** You set $p = 0.22$.
The simulated $\hat{p}$ will be **close** to $0.22$ when $n$ is large,
but not exactly equal, because the sample is random.

**Try it:** Change `n` to `40` and re-run. How far can $\hat{p}$ drift
from $p$ when the sample is small?

------------------------------------------------------------------------

## 2. Numerical outcomes: `rnorm` and population mean $\mu$ (and $\sigma$)

Continuous measurements (scores, heights, reaction times) are sometimes
modeled with a **Normal** universe in these labs: each observation is
**`rnorm(n, mean = mu, sd = sigma)`**.

- $\mu$ is the **population mean** (the `mean` argument in `rnorm`).
- $\sigma$ is the **population standard deviation** (the `sd` argument
  in `rnorm`).
- The **sample mean** $\bar{y} = \frac{1}{n}\sum_i y_i$ is computed with
  **`mean(y)`** in R. It only estimates $\mu$; it is **not** the same
  symbol as $\mu$.

In [ ]:
set.seed(1102)

mu <- 100 # population mean score
sigma <- 15 # population SD
n <- 200

y <- rnorm(n, mean = mu, sd = sigma)
y_bar <- mean(y)

tibble(
  population_mean_mu = mu,
  population_sd_sigma = sigma,
  sample_size = n,
  sample_mean_y_bar = y_bar
)

**Note:** **`rbinom`** is for **counts** out of a fixed number of trials
or for **binary** 0/1 data. It is **not** used for an arbitrary
continuous numerical outcome; that role belongs to **`rnorm`** (or other
models later in statistics).

------------------------------------------------------------------------

## 3. Two groups, **numerical** response: population means $\mu_1$ and $\mu_2$

You create a data frame with a **group** column and a **numerical** `y`.
Group 1 has $y \sim \mathcal{N}(\mu_1, \sigma)$ and group 2 has
$y \sim \mathcal{N}(\mu_2, \sigma)$. You can use the **same** $\sigma$
in both groups for simplicity (a teaching choice, not a rule about the
real world).

### 3A. Universe where **there is no difference** in population means

In [ ]:
set.seed(1103)

mu1 <- 50
mu2 <- 50 # same population mean in both groups
sigma <- 12
n_per <- 60

d_num_null <- tibble(
  group = factor(rep(c("A", "B"), each = n_per), levels = c("A", "B")),
  y = c(
    rnorm(n_per, mean = mu1, sd = sigma),
    rnorm(n_per, mean = mu2, sd = sigma)
  )
)

d_num_null |>
  group_by(group) |>
  summarise(sample_mean = mean(y), sample_sd = sd(y), n = n(), .groups = "drop")

**Hypotheses for Welch two-sample *t*-test** (what
`t.test(y ~ group, var.equal = FALSE)` uses by default):

- $H_0$: $\mu_1 = \mu_2$ (the **population mean** of $y$ is the same in
  group A and group B).
- $H_a$: $\mu_1 \neq \mu_2$ (two-sided by default).

Here $\mu_1$ is the mean of $y$ **among all units that would be labeled
A** in the population, and $\mu_2$ is the same for B.

In [ ]:
t.test(y ~ group, data = d_num_null, var.equal = FALSE)

**Interpretation template.** The *p*-value summarizes how compatible
these **sample** group means are with **“no difference in population
means”** ($H_0$) when random sampling is the only source of discrepancy.
Because you built the world with $\mu_1 = \mu_2$\`, a **large**
*p*-value is common; a **small** *p*-value can still happen by chance
(Type I error idea, Lecture 17).

### 3B. Universe where **there is a difference** in population means

In [ ]:
set.seed(1104)

mu1 <- 50
mu2 <- 58 # different population means
sigma <- 12
n_per <- 60

d_num_alt <- tibble(
  group = factor(rep(c("A", "B"), each = n_per), levels = c("A", "B")),
  y = c(
    rnorm(n_per, mean = mu1, sd = sigma),
    rnorm(n_per, mean = mu2, sd = sigma)
  )
)

t.test(y ~ group, data = d_num_alt, var.equal = FALSE)

**Try it:** Re-run 3A and 3B a few times with the same seeds removed
(delete `set.seed` lines or change seeds). Watch the *p*-value move even
when $H_0$ is true in 3A.

------------------------------------------------------------------------

## 4. Two groups, **binary** response: population proportions $p_1$ and $p_2$

For a binary outcome coded 0/1, $p_1$ is the population proportion of
successes in group 1 and $p_2$ in group 2.

### 4A. Universe where **there is no difference** in population proportions

In [ ]:
set.seed(1105)

p1 <- 0.18
p2 <- 0.18 # same population success probability
n_per <- 120

d_bin_null <- tibble(
  group = factor(rep(c("A", "B"), each = n_per), levels = c("A", "B")),
  y = c(
    rbinom(n_per, size = 1, prob = p1),
    rbinom(n_per, size = 1, prob = p2)
  )
)

counts_null <- d_bin_null |>
  group_by(group) |>
  summarise(successes = sum(y == 1), n = n(), p_hat = mean(y), .groups = "drop")

counts_null

**Hypotheses for `prop.test` comparing two proportions:**

- $H_0$: $p_1 = p_2$.
- $H_a$: $p_1 \neq p_2$ (two-sided default).

In [ ]:
prop.test(x = counts_null$successes, n = counts_null$n, correct = FALSE)

The output includes a **95% confidence interval** for $p_1 - p_2$
(difference in population proportions) and a *p*-value for the same
$H_0$.

### 4B. Universe where **there is a difference** in population proportions

In [ ]:
set.seed(1106)

p1 <- 0.18
p2 <- 0.32
n_per <- 120

d_bin_alt <- tibble(
  group = factor(rep(c("A", "B"), each = n_per), levels = c("A", "B")),
  y = c(
    rbinom(n_per, size = 1, prob = p1),
    rbinom(n_per, size = 1, prob = p2)
  )
)

counts_alt <- d_bin_alt |>
  group_by(group) |>
  summarise(successes = sum(y == 1), n = n(), p_hat = mean(y), .groups = "drop")

prop.test(x = counts_alt$successes, n = counts_alt$n, correct = FALSE)

**Check:** The sample proportions $\hat{p}_1$ and $\hat{p}_2$ should be
near `p1` and `p2`, and the *p*-value for $H_0: p_1 = p_2$ is often
**small** when $p_1 \neq p_2$ and $n$ is reasonably large.

------------------------------------------------------------------------

## 5. Cheat sheet: symbols in the population vs numbers from the sample

| Idea | Population (universe) | Sample (what you compute in R) |
|-------------------|-----------------------|-------------------------------|
| Binary one group | $p = P(Y=1)$ | $\hat{p} = \texttt{mean(y)}$ for 0/1 data |
| Numerical one group | $\mu$, $\sigma$ | $\bar{y} = \texttt{mean(y)}$, $s = \texttt{sd(y)}$ |
| Two numerical groups | $\mu_1$, $\mu_2$ | group means from `summarise(mean = mean(y))` |
| Two binary groups | $p_1$, $p_2$ | $\hat{p}_1$, $\hat{p}_2$ from group means of 0/1 |

Tests ask about population parameters, not sample quantities. For
example, the Welch *t*-test uses $\bar{y}_1$ and $\bar{y}_2$ to weigh
evidence about whether $\mu_1 - \mu_2$ is plausibly 0 after accounting
for sampling variability.

------------------------------------------------------------------------

## 6. A story: confounding with **no direct causal effect** of group on outcome

### Story (fiction)

The city piloted **“GreenSteps”** walking groups (A) versus usual
activity (B). Enrollment was **not randomized**: staff recruited more
heavily in neighborhoods that already had higher **walkability scores**
(a **categorical** summary: `walk` = `"easy"` vs `"hard"`). In our
simulation, **being assigned to a walking group does not change anyone’s
weekly steps** — only walkability changes steps. Still, a naive
comparison of steps by group can look like GreenSteps “works,” because
**walkability** is a **confounder**: it is associated with **both**
group and steps.

### Simulation: universe with **no effect** of `group` on `steps` given walkability

In [ ]:
set.seed(1107)

n <- 600

d_conf <- tibble(
  walk = factor(
    sample(c("hard", "easy"), size = n, replace = TRUE, prob = c(0.55, 0.45)),
    levels = c("hard", "easy")
  )
) |>
  mutate(
    # Group assignment depends on walkability (observational recruitment)
    p_green = if_else(walk == "easy", 0.62, 0.22),
    group = factor(
      ifelse(rbinom(n(), 1, prob = p_green) == 1, "GreenSteps", "Usual"),
      levels = c("Usual", "GreenSteps")
    ),
    # Weekly steps: depend ONLY on walkability (+ noise). No arrow from group.
    mu_steps = if_else(walk == "easy", 9800, 7200),
    steps = rnorm(n(), mean = mu_steps, sd = 900)
  )

**Crude comparison** (ignores walkability):

In [ ]:
t.test(steps ~ group, data = d_conf, var.equal = FALSE)

**Stratified description** (hold walkability fixed —
**subclassification**):

In [ ]:
d_conf |>
  group_by(walk, group) |>
  summarise(mean_steps = mean(steps), n = n(), .groups = "drop") |>
  arrange(walk, group)

What you should see. The crude *t*-test often suggests a difference in
population mean steps between GreenSteps and Usual. Within each level of
`walk`, however, mean steps for GreenSteps and Usual should be similar
(any small differences are just Monte Carlo noise), matching the fact
that in this universe group does not change `steps` — walkability drives
both recruitment and steps.

Conclusion in words. This is a toy virtual world where a resident who
only runs `t.test(steps ~ group)` can be misled about causation. Adding
the third variable and comparing within `walk` strata shows the
structure you imposed: confounding, not a treatment effect.